# Banc d'essai `bench` — génération AP-HP

Notebook du répertoire `generation/` (spec : `docs/spec_testrun_run_stage.md`),
suite courte d'appels au package `bench` : les fonctions de données sont dans
`bench/scenarios.py`, l'orchestration (gardes, idempotence, messages) dans
`bench/banc.py`. Cycle de vie d'un test : environnement → pool candidat →
montage → seeding/figement → génération → bilan. Toutes les étapes qui
touchent le disque sont **idempotentes** : elles s'exécutent si nécessaire,
sinon elles skippent avec un message d'état — le notebook se ré-exécute
intégralement sur un test déjà préparé. Rien ne se nettoie : pour repartir
de zéro, on crée `runs/NN+1`. Le disque fait foi ; **tout `generation/` est
versionné et partagé** (§9) — « figer » un jeu = commiter son test.

**Mode d'emploi.** Éditer la cellule **PARAMÈTRES COURANTS** (test courant,
parquet source, quotas, graine, `ONLY`), puis exécuter tout. Deux gestes à ne
pas confondre : le **montage** installe le jeu de templates du test dans
`system/one_gen/` depuis le test précédent (l'objet versionné) ; le
**seeding puis figement** crée les dossiers CRH depuis les scénarios, puis
résout le jeu par famille dans chaque dossier (la production du run).

- **Essai rapide d'un nouveau parquet** : `QUOTAS = "couverture"` (un
  scénario par type d'hospitalisation présent) et, au run réel,
  `ONLY = ["0000"]` (un seul appel payant). `preparer_pool` commence par le
  contrôle des prérequis du fichier (`verifier_source`) : colonnes, types,
  encodages, types d'hospitalisation affichés.
- **Itérer après édition du jeu** : changer `SYSTEM_PROMPT_FILE` et
  `OUT_FILE` (paramètres avancés) — les variantes coexistent, rien n'est
  écrasé. Le jeu s'édite dans `runs/<TEST_NUM>/system/one_gen/`.
- **Clé API** : exclusivement `MISTRAL_API_KEY` dans l'environnement du
  noyau, jamais dans le notebook. Tout s'exécute **sans clé** jusqu'aux
  dry-runs inclus ; seules les cellules « run réel » l'exigent.
- Annexes (2-gen, `prompt_local.py`, itération par copie, ajout de DAS) :
  `notebook_annexes.ipynb`.

In [2]:
# --- PARAMÈTRES COURANTS — la seule cellule éditée au quotidien ---
TEST_NUM = "06"    # le test courant (étiquettes tabac/alcool = données, bloc H)
PREV_TEST = "05"   # test précédent de la chaîne (None pour un tout premier test)

# Le parquet de scénarios (chemin depuis la racine du repo, ou absolu) — à
# garder IDENTIQUE d'un test à l'autre de la chaîne (comparabilité).
SOURCE_PROFILES_PATH = "data/aphp/scenarios_C1.parquet"

# Tirage : dict {modalité de QUOTAS_BY: effectif} (un quota à 0 documente une
# strate volontairement exclue ; la somme = la taille du test), OU
# "couverture" (1 scénario par type présent dans le fichier — le smoke d'un
# nouveau parquet), OU None (tirage simple de TARGET_N séjours).
QUOTAS = {
     "Séances simples": 2,
     "HDJ médecine adultes": 2,
     "Médecine adultes > 3 nuits": 2,
     "Chirurgie adultes < 3 nuits": 2,
     "Chirurgie adultes > 3 nuits": 2,
     "Interventionnel adultes < 3 nuits":1,
     "Interventionnel adultes > 3 nuits":1,
     "Accouchement normal mère": 1,
     "Bébé normal": 0,
     "IMG & fausses couches":1,
     "IVG": 0,  # DP en 8 : aucune IVG disponible
     "Greffes de moelle, CAR-T Cells": 0,
     "Brûlés" : 0,
     "Transplantations" : 0,
}
RANDOM_SEED = 42

ONLY = None  # ex. ["0000", "0001"] : run réel partiel (entrée `partial` au journal)


# --- PARAMÈTRES Modifiés
TEST_NUM = "07"
PREV_TEST = "06"
SOURCE_PROFILES_PATH = "data/aphp/scenarios_C1_dp.parquet"
QUOTAS = "couverture"      # un séjour par type : 15 scénarios
ONLY = ["0000"]            # un seul appel payant pour le premier run réel


In [3]:
# --- Bootstrap (ne pas toucher) : racine du repo, import de bench ---
import os
import sys
from pathlib import Path

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "bench").is_dir() and (p / "core").is_dir()), None)
assert REPO_ROOT, "Racine du repo Stream introuvable — lancer le notebook depuis generation/."
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import polars as pl
from bench import BenchError, Pricing, generate, load_reports, scenario_dirs
from bench.banc import *
from enrichissement import Politique

# --- PARAMÈTRES AVANCÉS (rarement touchés) ---
MODEL = os.environ.get("MISTRAL_MODEL", "mistral-large-latest")
MAX_TOKENS_SUMMARY = 8_000
MAX_TOKENS_CR = 128_000
# Transport des appels Mistral (spec v3.7) : "sync" = un chat.complete par
# scénario, MAX_WORKERS en parallèle ; "batch" = 50 % moins cher mais reste en
# file indéfiniment depuis septembre 2026 — à réactiver quand Mistral l'aura
# rétabli. Les tarifs (USD / 1M tokens, Mistral Large, https://mistral.ai/pricing)
# suivent le transport.
TRANSPORT = os.environ.get("MISTRAL_TRANSPORT", "sync")
MAX_WORKERS = 3
_TARIFS = {"sync": (0.5, 1.5), "batch": (0.25, 0.75)}
PRICING = Pricing(*_TARIFS[TRANSPORT])

# Sélection et enrichissement — à garder IDENTIQUES d'un test à l'autre de
# la chaîne (comparabilité).
QUOTAS_BY = "DPEC"          # ou "TPEC" — clés de QUOTAS
FILTRE_DP_SUFFIXE = "8"     # séjours dont le DP se termine par « 8 » (sous-
                            # catégories « autres formes précisées ») ; None = pas de filtre
TARGET_N = 15               # tirage simple, quand QUOTAS = None
SCENARIO_FILTERS: list[dict] = [
    # ex. {"column": "template_name", "op": "eq", "value": "surgery_outpatient.txt"},
]
RANDOM_SELECTION = True
ENRICHIR_SCENARIOS = True   # lot E1 : DAS tabac/alcool/corpulence + contexte patient
ENRICHISSEMENT_SEED = RANDOM_SEED
POLITIQUE_ENRICHISSEMENT = Politique()

# Fichiers du run — pour ITÉRER après édition du jeu : changer
# SYSTEM_PROMPT_FILE (ex. prompt_system_one_gen_v2.txt) et OUT_FILE (ex.
# crh_v2.txt) — les variantes coexistent, rien n'est écrasé.
SYSTEM_PROMPT_FILE = "prompt_system_one_gen.txt"
OUT_FILE = "crh_generation.txt"
PARAMS_GEN = dict(
    system=SYSTEM_PROMPT_FILE, user="user_generation.txt", out=OUT_FILE,
    model=MODEL, max_tokens=MAX_TOKENS_CR, pricing=PRICING, prefix_file="prefix.txt",
)

# Vérificateur (optionnel) : textes d'exemple, à adapter à la campagne.
VERIF_SYSTEM = """Tu es un médecin DIM. On te fournit un compte rendu
hospitalier généré automatiquement. Vérifie sa cohérence clinique et sa
conformité aux règles de codage, puis rends un verdict structuré :
CONFORME ou NON CONFORME, suivi de la liste des anomalies constatées."""
VERIF_USER = "Vérifie le compte rendu suivant et rends ton verdict."
VERIF_HEADER = "### COMPTE RENDU À VÉRIFIER"
VERIF_FOOTER = "### FIN DU COMPTE RENDU"
PARAMS_VERIF = dict(
    system="prompt_system_verif.txt", user="user_verification.txt", out="verdict.txt",
    model=MODEL, max_tokens=MAX_TOKENS_SUMMARY, pricing=PRICING,
    context_header=VERIF_HEADER, context_footer=VERIF_FOOTER,
)

pool = None  # rempli par preparer_pool (section 2)
print("Modèle :", MODEL)
print("Transport :", TRANSPORT, "— tarifs ($ / 1M tokens) :", PRICING)
print("MISTRAL_API_KEY présente :", "MISTRAL_API_KEY" in os.environ)

Modèle : mistral-large-latest
Transport : sync — tarifs ($ / 1M tokens) : Pricing(batch_input_usd_per_million=0.5, batch_output_usd_per_million=1.5)
MISTRAL_API_KEY présente : True


## 1. Environnement et pool candidat

`verifier_environnement()` : fictomed est l'éditable attendu (clone
`~/Documents/fictomed`, branche `prompt-work` — `uv sync` / `uv run`
réinstallent le paquet PyPI, wheel sans `regles_atih.yml` quelle que soit
sa version : refaire `uv pip install -e` puis
redémarrer le noyau), les bibliothèques de fiches respectent le contrat
recode-icd (`bench.fiches`, l'index fait foi) et le lecteur fictomed les lit.

`preparer_pool(...)` : contrôle des prérequis du parquet (`verifier_source`,
schéma `SCHEMA_SOURCE`), typologie TPEC/DPEC (conservée si le fichier la
fournit, sinon calculée), identifiants de traçabilité, filtre DP, tirage
(`QUOTAS`), enrichissement, contrôle du contrat côté pool (codes sans fiche
journalisés, codes ajoutés émissibles), couverture par type. Le pool sort
**prêt pour fictomed** — aucun appel Mistral ici.

In [4]:
verifier_environnement()

Racine repo  : /Users/remi/GitHub/Stream
bench        : /Users/remi/GitHub/Stream/bench
fictomed     : /Users/remi/Documents/fictomed/fictomed/__init__.py
Registre fictomed : 15111 fiches exactes, 2054 catégories (index : 15282 / 2097) — le lecteur lit la librairie.


(shape: (15_282, 15)
 ┌─────────┬─────────┬────────────┬────────────┬───┬────────────┬────────────┬──────────┬───────────┐
 │ code    ┆ chapter ┆ fichier    ┆ libelle    ┆ … ┆ source_exi ┆ classe_gen ┆ nb_chars ┆ format_ve │
 │ ---     ┆ ---     ┆ ---        ┆ ---        ┆   ┆ stence     ┆ eration    ┆ ---      ┆ rsion     │
 │ str     ┆ str     ┆ str        ┆ str        ┆   ┆ ---        ┆ ---        ┆ str      ┆ ---       │
 │         ┆         ┆            ┆            ┆   ┆ str        ┆ str        ┆          ┆ str       │
 ╞═════════╪═════════╪════════════╪════════════╪═══╪════════════╪════════════╪══════════╪═══════════╡
 │ A00.0   ┆ I       ┆ I/A00.0.md ┆ À Vibrio   ┆ … ┆ OWL_ANS    ┆ emissible  ┆ 1299     ┆ 1         │
 │         ┆         ┆            ┆ cholerae   ┆   ┆            ┆            ┆          ┆           │
 │         ┆         ┆            ┆ 01, biovar ┆   ┆            ┆            ┆          ┆           │
 │         ┆         ┆            ┆ c…         ┆   ┆         

In [5]:
pool = preparer_pool(
    SOURCE_PROFILES_PATH, QUOTAS, RANDOM_SEED,
    by=QUOTAS_BY, enrichir=ENRICHIR_SCENARIOS,
    enrichissement_seed=ENRICHISSEMENT_SEED, politique=POLITIQUE_ENRICHISSEMENT,
    filtre_dp_suffixe=FILTRE_DP_SUFFIXE, target_n=TARGET_N,
)

Source : /Users/remi/GitHub/Stream/data/aphp/scenarios_C1_dp.parquet
Shape  : (1087525, 38) — colonnes : ['branche', 'population', 'campagne', 'sexe', 'age', 'cage', 'mode_hospit', 'ghm2', 'racine', 'duree', 'nbda', 'type_unite', 'prep_sc', 'diag2', 'graine', 'diagnostic_associes', 'nb_das', 'diabete', 'diabete_scenario', 'hta', 'hta_scenario', 'mode_entree', 'mode_sortie', 'mdp', 'lettre', 'DPEC', 'TPEC', 'id_profil', 'id_scenario', 'hash_das', 'variante', 'poids', 'source_ref', 'nb_cible', 'nb_variantes_demandees', 'dp_origine', 'dp_substitue', 'repli_substitution']
fichier scenarios_C1_dp.parquet — 1087525 lignes, types d'hospitalisation (fournis) : {'Accouchement normal mère': 1744, 'Accouchement pathologique mère': 17523, 'Autre néonat': 287, 'Brûlés': 1557, 'Bébé normal': 627, 'Bébé néonat chir': 1186, 'Bébé néonat med': 26687, 'Chirurgie adultes < 3 nuits': 82527, 'Chirurgie adultes > 3 nuits': 104473, 'Greffes de moelle, CAR-T Cells': 75, 'HDJ médecine adultes': 71769, 'IMG & f

TPEC,DPEC,len
str,str,u32
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""",82527
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""",104473
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…",57594
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…",11000
"""Médecine""","""HDJ médecine adultes""",71769
…,…,…
"""Obstétrique""","""IMG & fausses couches""",8609
"""Obstétrique""","""IVG""",174
"""Séjours complexes""","""Brûlés""",1557


Filtre DP terminant par 8 : 1087525 -> 137332 séjours


TPEC,DPEC,len
str,str,u32
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""",7621
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""",2238
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…",7768
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…",564
"""Médecine""","""HDJ médecine adultes""",8644
…,…,…
"""Néonatalogie""","""Bébé néonat med""",4147
"""Obstétrique""","""Accouchement normal mère""",131
"""Obstétrique""","""Accouchement pathologique mère""",1418


Séjours incomplets pour fictomed (agean / duree / mode_entree / mode_sortie nuls) : 13588 écarté(s) — 137332 -> 123744 séjours
Couverture : 1 séjour par modalité de DPEC présente (15 modalité(s)).
Tirage stratifié par DPEC : 15 séjours (15 strate(s), seed=42).


...

TPEC,DPEC,specialty,specialite_source,len
str,str,str,str,u32
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""","""GYNECOLOGIE""","""unique""",1
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""","""CHIR.PLAST.RECONSTR.""","""unique""",1
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…","""HEPATO-GASTRO-ENTERO""","""unique""",1
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…","""CARDIOLOGIE""","""unique""",1
"""Médecine""","""HDJ médecine adultes""","""MEDECINE INTERNE""","""unique""",1
…,…,…,…,…
"""Néonatalogie""","""Bébé néonat med""","""NEONATOLOGIE""","""observee""",1
"""Obstétrique""","""Accouchement normal mère""","""OBSTETRIQUE""","""unique""",1
"""Obstétrique""","""Accouchement pathologique mère""","""OBSTETRIQUE""","""unique""",1


Enrichissement : 9/15 ligne(s) enrichie(s), 6 exclue(s), 6 code(s) DAS ajouté(s).
Exclusions : préfixe O ×3, âge < 18 (ou manquant) ×3
Codes ajoutés : E6603×3, F101×1, E6604×1, F1725×1


DPEC,agean,sexe,taille_cm,poids_kg,imc,tabac,alcool,codes_ajoutes
str,i32,str,i64,i64,f64,str,str,str
"""Brûlés""",26,"""1""",189,79,22.1,"""non""","""environ 5 verres/jour""","""F101"""
"""Chirurgie adultes < 3 nuits""",46,"""2""",171,71,24.3,"""non""","""non""",""""""
"""Chirurgie adultes > 3 nuits""",68,"""2""",159,59,23.3,"""non""","""non""",""""""
"""HDJ médecine adultes""",63,"""2""",159,87,34.4,"""non""","""non""","""E6604"""
"""Interventionnel adultes < 3 nu…",58,"""2""",164,68,25.3,"""non""","""non""","""E6603"""
"""Interventionnel adultes > 3 nu…",84,"""2""",155,52,21.6,"""actif, 12 cigarettes/jour, 36 …","""non""","""F1725"""
"""Médecine adultes < 3 nuits""",68,"""2""",164,69,25.7,"""non""","""non""","""E6603"""
"""Médecine adultes > 3 nuits""",82,"""2""",154,72,30.4,"""non""","""non""",""""""
"""Séances simples""",63,"""2""",166,72,26.1,"""non""","""non""","""E6603"""


Pool : 75 codes distincts, 1 sans fiche à l'index — JOURNAL : ['X3100']
Enrichissement : 4 code(s) ajouté(s) distincts, tous émissibles.
Pool candidat : 15 séjours — agean dérivé de cage, pivot age — racine réparée sur 372108 ligne(s) — 13588 séjour(s) incomplet(s) écarté(s) avant tirage — couverture par type :


TPEC,DPEC,len
str,str,u32
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""",1
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""",1
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…",1
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…",1
"""Médecine""","""HDJ médecine adultes""",1
…,…,…
"""Néonatalogie""","""Bébé néonat med""",1
"""Obstétrique""","""Accouchement normal mère""",1
"""Obstétrique""","""Accouchement pathologique mère""",1


## 2. Test courant — montage, seeding puis figement

**Le jeu du test s'édite LÀ : `runs/<TEST_NUM>/system/one_gen/` — un `.txt`
par famille clinique.** Il est monté par copie du jeu du test précédent (la
chaîne des tests est la chaîne des versions) ; pour un tout premier test,
fournir un jeu initial à la main (les jeux historiques sont dans git :
`generation/runs/01`). Un `prefix.txt` dans le jeu remplace le prefix
fictomed au seeding ; quand `ENRICHIR_SCENARIOS` est actif, le contexte
patient (taille, poids, IMC, tabac, alcool) est fourni dans le user prompt et
le prompt système doit le restituer (bloc H du jeu de `runs/05`).

`seeder` : génération fictomed (un scénario par ligne du pool ; skip si le
test est déjà seedé), graine (dossiers CRH, `test.json` annoté), puis
figement du jeu par famille en `SYSTEM_PROMPT_FILE` (skip si déjà figé).

In [6]:
TD, PREV_TD = dossiers_test(TEST_NUM, PREV_TEST)
etat_test(TD, PREV_TD)
monter_jeu(TD, PREV_TD)

Test courant : /Users/remi/GitHub/Stream/generation/runs/07 (existe)
Jeu amont    : /Users/remi/GitHub/Stream/generation/runs/06/system/one_gen
Test               : /Users/remi/GitHub/Stream/generation/runs/07 — présent
Jeu system/one_gen : présent
Dossiers scénario  : 15 ['0000', '0001', '0002', '0003', '0004'] …
Figement (1er dossier, prompt_system_one_gen.txt) : présent
SKIP — jeu déjà monté : /Users/remi/GitHub/Stream/generation/runs/07/system/one_gen


In [7]:
selected_scenarios = seeder(
    TD, pool, source_path=SOURCE_PROFILES_PATH,
    enrichir=ENRICHIR_SCENARIOS, enrichissement_seed=ENRICHISSEMENT_SEED,
    politique=POLITIQUE_ENRICHISSEMENT, seed=RANDOM_SEED,
    scenario_filters=SCENARIO_FILTERS, random_selection=RANDOM_SELECTION,
    system_prompt_file=SYSTEM_PROMPT_FILE,
)

SKIP — test déjà seedé : 15 dossiers scénario.


TPEC,DPEC,generation_id,template_name,case_management_type,age,sexe,icd_primary_description,icd_primary_code,nb_associated,department,specialite_source,taille_cm,poids_kg,imc,tabac,alcool,codes_ajoutes
str,str,str,str,str,i64,i64,str,str,i64,str,str,i64,i64,f64,str,str,str
"""Médecine""","""Séances simples""","""a46e798b-7795-4de7-91ce-427dfb…","""medical_outpatient.txt""","""Z512""",63,2,"""Autres ostéoporoses - Autres l…","""M8188""",null,null,"""repli""",166,72,26.1,"""non""","""non""","""E6603"""
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""","""eae727d3-fef3-4917-b358-8779dc…","""surgery_outpatient.txt""","""Z52801""",46,2,"""Stérilité de la femme d'autres…","""N978""",null,"""GYNECOLOGIE""","""unique""",171,71,24.3,"""non""","""non""",""""""
"""Néonatalogie""","""Autre néonat""","""05b83d30-64a4-4d5b-bf19-38f2e3…","""medical_inpatient.txt""","""DP""",0,2,"""Autres détresses respiratoires…","""P228""",null,"""NEONATOLOGIE""","""unique""",null,null,null,null,null,null
"""Obstétrique""","""Accouchement pathologique mère""","""59136c52-b64b-46e9-9466-4bc406…","""delivery_inpatient_urg.txt""","""DP""",32,2,"""Travail et accouchement compli…","""O698""",8,"""OBSTETRIQUE""","""unique""",null,null,null,null,null,null
"""Néonatalogie""","""Bébé normal""","""29c3e854-de9f-4930-89a2-c8333c…","""medical_inpatient.txt""","""DP""",0,1,"""Ictère néonatal dû à d'autres …","""P598""",4,"""NEONATOLOGIE""","""observee""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Séjours complexes""","""Brûlés""","""424c1b40-90bc-46b6-a065-4e250f…","""medical_inpatient.txt""","""DP""",26,1,"""Gelure de la cheville et du pi…","""T348""",2,null,"""repli""",189,79,22.1,"""non""","""environ 5 verres/jour""","""F101"""
"""Médecine""","""HDJ médecine adultes""","""7d2fbe77-3550-483e-b4ca-91838d…","""medical_outpatient.txt""","""DP""",63,2,"""Autres anémies précisées""","""D648""",null,"""MEDECINE INTERNE""","""unique""",159,87,34.4,"""non""","""non""","""E6604"""
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""","""f223ec6e-1bb6-46a7-88bb-9aea4c…","""surgery_inpatient.txt""","""Z421""",68,2,"""Autres affections précisées du…","""N648""",4,"""CHIR.PLAST.RECONSTR.""","""unique""",159,59,23.3,"""non""","""non""",""""""


SKIP — prompt_system_one_gen.txt déjà figé dans tous les dossiers.


## 3. Génération — contrôle à sec puis run réel

Le contrôle à sec ne fait aucun appel API et aucune écriture — pas besoin de
clé. C'est aussi le **test de complétude** : `generate` échoue (`BenchError`)
au moindre fichier manquant, aucun dossier n'est sauté en silence. Le run
réel écrase `OUT_FILE` à chaque re-run (geste normal) ; chaque run réel
ajoute son entrée au journal `usage.json` (et une ligne par scénario au
journal CSV global `generation/usage_log.csv`). `ONLY` restreint le run réel
à quelques dossiers (entrée `partial` au journal).

In [7]:
dry = generate(TD, client=None, dry_run=True, **PARAMS_GEN)  # inutile de fournir un client
show_first_prompt(dry)

Scénario : 0000 — famille : medical_outpatient

============================ PROMPT SYSTÈME — 17258 caractère(s) ============================
Vous êtes un médecin clinicien expert. Votre tâche est de générer un compte rendu d'hospitalisation détaillé à partit d'un scénario clinique réalisé avec des codes de la classification internationale des maladies et d'autres informations décrivant l'hospitalisation.


# Contexte : le codage CIM-10

La CIM-10 est une classification des maladies, elle peut se définir comme un ensemble organisé de rubriques dans lesquelles on range des entités morbides en fonction de certains critères établis. La CIM est utilisée pour transposer les diagnostics de maladies ou autres problèmes de santé en codes alphanumériques, ce qui facilite le stockage, la recherche et l'analyse des données. Elle est très utilisée en France, en particulier pour le codage des causes de décès et pour la déclaration de l'activité hospitalière dans le cadre du programme de médicalisat

In [14]:
client = mistral_client()  # échoue ici, clairement, si MISTRAL_API_KEY absente
cr = generate(
    TD, client=client, transport=TRANSPORT, max_workers=MAX_WORKERS,
    only=ONLY, **PARAMS_GEN,
)
print(cr.usage)

Run Mistral sync_20260925_123004_005132 — 15 requête(s) synchrone(s), modèle mistral-large-latest, 3 appel(s) en parallèle, JSONL : /Users/remi/GitHub/Stream/generation/runs/07/batches/crh_generation/sync_input_20260925_123004_005132.jsonl
  1/15 — 0001 : ok
  2/15 — 0000 : ok
  3/15 — 0002 : ok
  4/15 — 0003 : ok
  5/15 — 0004 : ok
  6/15 — 0005 : ok
  7/15 — 0006 : ok
  8/15 — 0008 : ok
  9/15 — 0007 : ok
  10/15 — 0010 : ok
  11/15 — 0011 : ok
  12/15 — 0009 : ok
  13/15 — 0012 : ok
  14/15 — 0013 : ok
  15/15 — 0014 : ok
Usage(n_requests=15, input_tokens=121232, output_tokens=29507, total_tokens=150739, input_cost_usd=0.060616, output_cost_usd=0.044260499999999994, total_cost_usd=0.1048765)


In [19]:
# Lecture des CR : rendu markdown inline du premier scénario, puis un aperçu
# .md à côté de chaque .txt du test (ex. 0007/crh_generation.md, versionné —
# « Markdown: Open Preview », ⇧⌘V). afficher_crh(TD, "0007") pour un autre.
CRH = "0005"   # numéro du dossier à relire, ou None pour le premier
_names = scenario_dirs(TD)
if _names:
    afficher_crh(TD, CRH or _names[0], OUT_FILE)
else:
    print("Pas encore de dossiers scénario dans", TD)
ecrire_apercus_md(TD, OUT_FILE)

### Compte rendu d'hospitalisation

**Hôpital Saint-Louis - Assistance Publique Hôpitaux de Paris**
**Service de CARDIOLOGIE**

**Nom :** Parant
**Prénom :** Jeannette
**Date de naissance :** 04/03/1942
**Âge :** 84 ans

---

### Motif d’hospitalisation
Mme Jeannette Parant, 84 ans, a été hospitalisée le 30/11/2026 pour exploration et prise en charge d’une douleur thoracique récurrente, survenant à l’effort et parfois au repos, associée à une sensation d’oppression. Ces symptômes évoquent un angor d’effort, dans un contexte d’hypertension artérielle connue et de fibrillation auriculaire paroxystique.

---

### Antécédents
- **Médicaux :**
  - Hypertension artérielle essentielle, suivie depuis plusieurs années.
  - Fibrillation auriculaire paroxystique, connue depuis 2020.
  - Sténose aortique calcifiée, non rhumatismale, diagnostiquée en 2022.
  - Hypothyroïdie, sous traitement substitutif.
  - Sinusite maxillaire chronique, évoluant depuis plusieurs années.
  - Dénutrition modérée, objectivée par un IMC à 21,6 kg/m² et une perte de poids récente.
  - Tabagisme actif (12 cigarettes/jour, 36 paquets-années).
- **Chirurgicaux :** Aucun antécédent chirurgical notable.
- **Familiaux :** Pas d’antécédent familial de cardiopathie ischémique ou de mort subite.
- **Allergies :** Aucune allergie médicamenteuse connue.

---

### Mode de vie
Mme Parant est retraitée, vivant à domicile de manière autonome. Elle fume activement 12 cigarettes par jour depuis plus de 50 ans (36 paquets-années). Elle ne consomme pas d’alcool. Son alimentation est déséquilibrée, avec des apports protéino-énergétiques insuffisants, contribuant à sa dénutrition modérée.

---

### Histoire de la maladie
Depuis quelques semaines, Mme Parant présente des épisodes de douleur thoracique constrictive, survenant initialement à l’effort (montée d’escaliers, marche rapide) et plus récemment au repos. Ces symptômes s’accompagnent d’une sensation d’oppression thoracique et de palpitations. Elle a consulté son médecin traitant, qui a suspecté un angor d’effort dans un contexte d’hypertension artérielle mal contrôlée et de fibrillation auriculaire paroxystique. Une hospitalisation a été décidée pour exploration coronarienne et ajustement thérapeutique.

---

### Examen clinique
À l’admission, Mme Parant est apyrétique, avec une tension artérielle à 160/90 mmHg, une fréquence cardiaque irrégulière à 88 battements/minute, et une saturation en oxygène à 96 % en air ambiant. Son poids est de 52 kg pour une taille de 155 cm (IMC = 21,6 kg/m²). L’auscultation cardiaque révèle un souffle systolique éjectionnel au foyer aortique, irradiant vers les carotides, compatible avec une sténose aortique serrée. Les pouls périphériques sont présents et symétriques. L’examen pulmonaire est sans particularité, en dehors d’une légère diminution du murmure vésiculaire à droite, évoquant une séquelle de sinusite maxillaire chronique. L’abdomen est souple, sans hépatomégalie ni signe d’insuffisance cardiaque droite. Les membres inférieurs ne présentent pas d’œdème.

---

### Examens complémentaires
- **Biologie :**
  - NFS : légère anémie normocytaire (Hb = 11,2 g/dL), sans anomalie des leucocytes ni des plaquettes.
  - Ionogramme sanguin : kaliémie à 4,1 mmol/L, natrémie à 138 mmol/L.
  - Créatinine : 102 µmol/L (DFG estimé à 45 mL/min/1,73 m²).
  - Bilan hépatique : normal.
  - TSH : 6,2 mUI/L (sous traitement substitutif), confirmant un équilibre suboptimal de l’hypothyroïdie.
  - Gaz du sang artériel : légère hypoxémie (PaO₂ = 72 mmHg) avec normocapnie.
  - Troponine ultrasensible : négative à deux reprises.
- **ECG :** Fibrillation auriculaire à réponse ventriculaire rapide (90 battements/minute), sans signe d’ischémie aiguë.
- **Échocardiographie transthoracique :** Sténose aortique serrée (surface aortique estimée à 0,8 cm²), avec un gradient moyen transvalvulaire à 45 mmHg. Fraction d’éjection ventriculaire gauche conservée (60 %). Légère hypertrophie ventriculaire gauche.
- **Coronarographie (01/12/2026) :** Réalisée par voie radiale. Mise en évidence d’une sténose serrée de l’artère interventriculaire antérieure proximale (70 %), ainsi que de lésions modérées sur la coronaire droite et la circonflexe. Une ventriculographie gauche a confirmé la bonne fonction systolique globale.

---

### Évolution pendant l'hospitalisation
#### Démarche diagnostique
L’exploration coronarienne a permis de confirmer le diagnostic d’angor d’effort, secondaire à une maladie coronarienne athéromateuse. La sténose aortique serrée, déjà connue, a été réévaluée et jugée non chirurgicale à ce stade, en raison de l’âge et des comorbidités. La fibrillation auriculaire paroxystique a été documentée à plusieurs reprises, sans épisode prolongé nécessitant une cardioversion.

#### Traitements
- **Traitement médicamenteux :**
  - Introduction d’un bêta-bloquant (bisoprolol 2,5 mg/jour) pour ralentir la fréquence cardiaque et réduire la consommation myocardique en oxygène.
  - Poursuite du traitement antihypertenseur (inhibiteur calcique et diurétique thiazidique).
  - Optimisation du traitement de l’hypothyroïdie (augmentation de la lévothyroxine à 75 µg/jour).
  - Introduction d’un antiagrégant plaquettaire (clopidogrel 75 mg/jour) en prévention secondaire.
  - Anticoagulation par antivitamine K (warfarine, INR cible 2-3) pour la fibrillation auriculaire.
  - Supplémentation nutritionnelle (compléments hyperprotéinés) pour corriger la dénutrition modérée.
- **Prise en charge des comorbidités :**
  - Consultation ORL pour évaluation de la sinusite maxillaire chronique, avec proposition d’un traitement local (lavages de nez).
  - Sevrage tabagique initié, avec proposition d’un suivi spécialisé en consultation de tabacologie.
  - Surveillance régulière de la fonction rénale et de l’équilibre thyroïdien.

Mme Parant a bien toléré les explorations et les ajustements thérapeutiques. Les douleurs thoraciques ont disparu sous traitement médical. Elle a été éduquée sur les signes d’alerte nécessitant une consultation en urgence (douleur thoracique, dyspnée, palpitations).

---

### Conclusion
Mme Jeannette Parant, 84 ans, a été hospitalisée pour exploration d’un angor d’effort dans un contexte d’hypertension artérielle, de fibrillation auriculaire paroxystique et de sténose aortique serrée. La coronarographie a révélé une sténose significative de l’artère interventriculaire antérieure, justifiant une prise en charge médicamenteuse optimisée. La dénutrition modérée et l’hypothyroïdie ont également été prises en charge. Elle sort ce jour (03/12/2026) avec un traitement adapté et un suivi rapproché en cardiologie.

**Dr Fernando Brando**
Service de Cardiologie
Hôpital Saint-Louis - AP-HP


---

## Formulations

```json
{
  "diagnostics": {
    "Autres formes d'angine de poitrine (I20.8)": [
      "angor d’effort",
      "angor d’effort dans un contexte d’hypertension artérielle mal contrôlée et de fibrillation auriculaire paroxystique",
      "angor d’effort, secondaire à une maladie coronarienne athéromateuse"
    ],
    "Hypertension essentielle (primitive) (I10)": [
      "hypertension artérielle essentielle",
      "hypertension artérielle mal contrôlée",
      "hypertension artérielle"
    ],
    "Malnutrition protéino-énergétique modérée (E44.0)": [
      "dénutrition modérée",
      "apports protéino-énergétiques insuffisants, contribuant à sa dénutrition modérée"
    ],
    "Douleur thoracique, sans précision (R07.4)": [
      "douleur thoracique récurrente",
      "douleur thoracique constrictive",
      "sensation d’oppression thoracique"
    ],
    "Fibrillation auriculaire paroxystique (I48.0)": [
      "fibrillation auriculaire paroxystique",
      "fibrillation auriculaire paroxystique, connue depuis 2020"
    ],
    "Autres résultats anormaux précisés des examens chimiques du sang (R79.8)": [
      "légère hypoxémie (PaO₂ = 72 mmHg) avec normocapnie",
      "TSH : 6,2 mUI/L (sous traitement substitutif)"
    ],
    "Hypothyroïdie, sans précision (E03.9)": [
      "hypothyroïdie",
      "équilibre suboptimal de l’hypothyroïdie"
    ],
    "Sténose (de la valvule) aortique (non rhumatismale) (I35.0)": [
      "sténose aortique calcifiée, non rhumatismale",
      "sténose aortique serrée",
      "rétrécissement aortique calcifié serré"
    ],
    "Sinusite maxillaire (chronique) (J32.0)": [
      "sinusite maxillaire chronique",
      "séquelle de sinusite maxillaire chronique"
    ],
    "Dépendance envers un respirateur : ventilation par masque nasal (Z99.1+1)": [],
    "Syndrome de dépendance au tabac, utilisation continue (F17.25)": [
      "tabagisme actif (12 cigarettes/jour, 36 paquets-années)"
    ]
  },
  "informations": {
    "Date entrée": [
      "30/11/2026"
    ],
    "Date de sortie": [
      "03/12/2026"
    ],
    "Service d'hospitalisation": [
      "CARDIOLOGIE"
    ],
    "Nom/Prénom du patient": [
      "Parant Jeannette"
    ],
    "Nom/Prénom du médecin": [
      "Fernando Brando"
    ],
    "Âge": [
      "84 ans"
    ],
    "Sexe": [
      "Féminin"
    ],
    "État général": [
      "Mme Parant est apyrétique, avec une tension artérielle à 160/90 mmHg, une fréquence cardiaque irrégulière à 88 battements/minute, et une saturation en oxygène à 96 % en air ambiant."
    ],
    "Poids": [
      "52 kg pour une taille de 155 cm (IMC = 21,6 kg/m²)"
    ],
    "Statut gestationnel": [],
    "Gestité": [],
    "NFS": [
      "légère anémie normocytaire (Hb = 11,2 g/dL), sans anomalie des leucocytes ni des plaquettes"
    ],
    "Créatinine": [
      "102 µmol/L (DFG estimé à 45 mL/min/1,73 m²)"
    ],
    "Bilan hépatique": [
      "normal"
    ],
    "Traitements": [
      "Introduction d’un bêta-bloquant (bisoprolol 2,5 mg/jour)",
      "Poursuite du traitement antihypertenseur (inhibiteur calcique et diurétique thiazidique)",
      "Optimisation du traitement de l’hypothyroïdie (augmentation de la lévothyroxine à 75 µg/jour)",
      "Introduction d’un antiagrégant plaquettaire (clopidogrel 75 mg/jour)",
      "Anticoagulation par antivitamine K (warfarine, INR cible 2-3)",
      "Supplémentation nutritionnelle (compléments hyperprotéinés)"
    ]
  }
}
```


/Users/remi/GitHub/Stream/generation/runs/07/0000/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0001/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0002/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0003/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0004/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0005/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0006/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0007/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0008/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0009/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0010/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0011/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0012/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/0013/crh_generation.md
/Users/remi/GitHub/Stream/generation/runs/07/001

## 4. Vérificateur (optionnel) et bilan

Verdict nourri par les CR de la section 3 — depuis le disque (`load_reports`,
le disque fait foi ; `strict=False` charge les scénarios déjà servis et
`only=` restreint le run à ceux-là). Les prompts partagés du vérificateur sont
posés une fois (`write_prompts`) ; le contrôle à sec accepte un contexte
placeholder sans run réel.

Bilan : journal **append-only** des coûts (`committed_usd` = total engagé,
`current_usd` = coût de l'état courant), stats du journal CSV global, puis
les contrôles mécaniques hors modèle (`scripts/check_crh.py`,
`scripts/reancre_crh.py`, stdlib uniquement).

In [8]:
prompts_verificateur(TD, VERIF_SYSTEM, VERIF_USER)
ctx = contexte_verificateur(TD, OUT_FILE)
dry_verif = generate(TD, client=None, dry_run=True, context=ctx, **PARAMS_VERIF)
show_first_prompt(dry_verif)

SKIP — prompt_system_verif.txt déjà présent dans tous les dossiers.
SKIP — user_verification.txt déjà présent dans tous les dossiers.
Scénario : 0000 — famille : medical_outpatient

============================ PROMPT SYSTÈME — 260 caractère(s) ============================
Tu es un médecin DIM. On te fournit un compte rendu
hospitalier généré automatiquement. Vérifie sa cohérence clinique et sa
conformité aux règles de codage, puis rends un verdict structuré :
CONFORME ou NON CONFORME, suivi de la liste des anomalies constatées.

============================ PROMPT USER — 5560 caractère(s) ============================
Vérifie le compte rendu suivant et rends ton verdict.

### COMPTE RENDU À VÉRIFIER

{"CR": "### Hôpital Haut-Lévêque - CHU de Bordeaux\nService d'hospitalisation : Hospitalisation ambulatoire\n\n**Nom :** Schambil\n**Prénom :** Claudia\n**Date de naissance :** 10/06/1963\n**Âge :** 63 ans\n\n---

### Motif d'hospitalisation
Patiente de 63 ans adressée pour une séance de c

In [9]:
client = mistral_client()

cr_disque = load_reports(TD, OUT_FILE, strict=False)
if cr_disque.height == 0:
    raise RuntimeError(f"Aucun {OUT_FILE} sur disque : lancer d'abord le run réel (section 3).")

verdicts = generate(
    TD, client=client, transport=TRANSPORT, max_workers=MAX_WORKERS,
    context=cr_disque, only=cr_disque["scenario"].to_list(), **PARAMS_VERIF,
)
print(verdicts.usage)

Run Mistral sync_20260925_152932_367142 — 15 requête(s) synchrone(s), modèle mistral-large-latest, 3 appel(s) en parallèle, JSONL : /Users/remi/GitHub/Stream/generation/runs/07/batches/verdict/sync_input_20260925_152932_367142.jsonl
  1/15 — 0001 : ok
  2/15 — 0000 : ok
  3/15 — 0002 : ok
  4/15 — 0004 : ok
  5/15 — 0003 : ok
  6/15 — 0007 : ok
  7/15 — 0006 : ok
  8/15 — 0005 : ok
  9/15 — 0008 : ok
  10/15 — 0010 : ok
  11/15 — 0009 : ok
  12/15 — 0011 : ok
  13/15 — 0013 : ok
  14/15 — 0012 : ok
  15/15 — 0014 : ok
Usage(n_requests=15, input_tokens=31074, output_tokens=23470, total_tokens=54544, input_cost_usd=0.015537, output_cost_usd=0.035205, total_cost_usd=0.050742)


In [10]:
bilan(TD, out_file=OUT_FILE)

=== 01 — /Users/remi/GitHub/Stream/generation/runs/01 ===


out,n_runs,committed_usd,current_usd
str,i64,f64,f64
"""crh_generation.txt""",10,0.090834,0.045261
"""TOTAL""",10,0.090834,0.045261


=== 02 — /Users/remi/GitHub/Stream/generation/runs/02 ===


out,n_runs,committed_usd,current_usd
str,i64,f64,f64
"""crh_generation.txt""",2,0.0647135,0.033556
"""verdict.txt""",1,0.015932,0.015932
"""TOTAL""",3,0.080646,0.049488


=== 03 — /Users/remi/GitHub/Stream/generation/runs/03 ===


...

generation/usage_log.csv — 45 ligne(s)
Coût et tokens par (test, out) :


test,out,n_lignes,input_tokens,output_tokens,cost_usd
str,str,u32,i64,i64,f64
"""06""","""crh_generation.txt""",14,100625,23338,0.085318
"""07""","""crh_generation.txt""",16,129106,31258,0.11144
"""07""","""verdict.txt""",15,31074,23470,0.05074


Test courant 07 — par template :


template,n_lignes,input_tokens,output_tokens,cost_usd
str,u32,i64,i64,f64
"""delivery_inpatient_hospit""",2,11587,4524,0.012579
"""delivery_inpatient_urg""",2,12324,4053,0.012242
"""medical_inpatient""",14,76539,25741,0.07688
"""medical_outpatient""",7,33350,11477,0.03389
"""surgery_inpatient""",4,20956,7315,0.02145
"""surgery_outpatient""",2,5424,1618,0.005139


[0000]
    ECHEC  gras interdit (11 occurrence(s)) : **Nom :**, **Prénom :**, **Date de naissance :**, **Âge :**
    ECHEC  formulation fantôme [diagnostics/Autres ostéoporoses - Autres localisations (M81.88)] : « ostéoporose sénile avec atteinte vertébrale dorsale et lomba »
    ECHEC  formulation fantôme [diagnostics/Diabète sucré de type 2 insulinotraité, avec complications oculaires (E11.30)] : « diabète de type 2 insulinotraité, compliqué d'une rétinopath »
    ECHEC  formulation fantôme [diagnostics/Surpoids dû à un excès calorique, de l'adulte ou de l'enfant (E66.03)] : « surpoids modéré »
    ECHEC  formulation fantôme [informations/Date entrée] : « 12/08/2026 »
    ECHEC  formulation fantôme [informations/Date de sortie] : « 12/08/2026 »
    ECHEC  formulation fantôme [informations/Nom/Prénom du patient] : « Schambil Claudia »
    ECHEC  formulation fantôme [informations/Sexe] : « Féminin »
    AVERT  mention IPP absente de l'en-tête
    AVERT  maladie chronique mentionnée : d

### Préparer les entrées du juge (optionnel)

Après un run réel : le nettoyage d'export (`scripts/nettoie_dictionnaire.py`,
sortie `export_dict/` du test) ne garde du dictionnaire de formulations que
les passages présents verbatim dans le CR ; puis `juge_io.ecrire_entrees_juge`
écrit `entrees_juge.jsonl` — une ligne par code du scénario (code, libellé,
fiche, passages propres). Ces entrées servent aujourd'hui aux **verdicts
manuels** (preuve directe du diagnostic dans le texte, oui/non) et demain à
l'**encodeur** qui les rendra automatiquement ; leurs verdicts nourrissent
`prepare_regeneration`. Le contrat d'entrée et de sortie fait foi dans la
docstring de `scripts/juge_io.py`.


In [11]:
entrees_juge = preparer_entrees_juge(TD, out_file=OUT_FILE)


[0000]
    bilan : 17 gardée(s), 3 réancrée(s), 4 supprimée(s)
[0001]
    bilan : 7 gardée(s), 0 réancrée(s), 3 supprimée(s)
[0002]
    bilan : 14 gardée(s), 2 réancrée(s), 0 supprimée(s)
[0003]
    bilan : 32 gardée(s), 3 réancrée(s), 1 supprimée(s)
[0004]
    bilan : 22 gardée(s), 2 réancrée(s), 2 supprimée(s)
[0005]
    bilan : 34 gardée(s), 4 réancrée(s), 3 supprimée(s)
[0006]
    bilan : 17 gardée(s), 4 réancrée(s), 0 supprimée(s)
[0007]
    bilan : 36 gardée(s), 14 réancrée(s), 3 supprimée(s)
[0008]
    bilan : 11 gardée(s), 2 réancrée(s), 2 supprimée(s)
[0009]
    bilan : 30 gardée(s), 6 réancrée(s), 0 supprimée(s)
[0010]
    bilan : 27 gardée(s), 0 réancrée(s), 2 supprimée(s)
[0011]
    bilan : 24 gardée(s), 4 réancrée(s), 1 supprimée(s)
[0012]
    bilan : 18 gardée(s), 5 réancrée(s), 4 supprimée(s)
[0013]
    bilan : 21 gardée(s), 3 réancrée(s), 4 supprimée(s)
[0014]
    bilan : 30 gardée(s), 1 réancrée(s), 1 supprimée(s)

Bilan global : 393 formulation(s) exportée(s), 30 supp

# Annexe — Lire le bilan d'un test : le lexique des contrôles

*Annexe d'information pour la section 4 du notebook. Ce qui fait foi reste
les scripts (`scripts/check_crh.py`, `scripts/reancre_crh.py`,
`scripts/juge_io.py`) et leurs docstrings. Exemples tirés du test 07.*

---

## Le principe : deux objets vérifiés, trois étages de lecture

Chaque génération produit **deux objets** :

1. **le CRH** — le texte du compte rendu ;
2. **le dictionnaire d'annotation** — pour chaque code CIM-10 (section
   *diagnostics*) et chaque donnée du scénario (section *informations*),
   le modèle cite **l'extrait exact du texte** qui le prouve. C'est ce
   dictionnaire qui fera du corpus un jeu d'entraînement annoté.

Le bilan se lit en **trois étages**, et un chiffre brut ne se lit jamais
sans son étage de traitement :

| Étage | Outil | Question posée |
|---|---|---|
| 1. Contrôles | `check_crh` | le CRH et son dictionnaire respectent-ils le contrat ? |
| 2. Réancrage | `reancre_crh` | les citations inexactes sont-elles récupérables ? |
| 3. Jugement | entrées juge (`juge_io`) | chaque code a-t-il une **vraie preuve** dans le texte ? |

---

## Étage 1 — les sorties de `check_crh`

### Les ECHEC

**`gras interdit (N occurrences)`**
Le CRH contient du texte entre `**…**` (mise en forme markdown), interdit
car un vrai CRH est du texte brut. Presque toujours des étiquettes de
structure (`**Nom :**`, `**Médicaux :**`).
→ *Destination : nettoyage d'export (suppression mécanique). Jamais de
régénération pour ça — zéro coût qualité.*

**`formulation fantôme [section/clé] : « extrait »`**
L'extrait cité par le dictionnaire **ne se trouve pas mot pour mot** dans
le texte : le modèle a cité de mémoire en reformulant. Exemple (07/0014) :
cité « tachycardie à 110/min », écrit « tachycardie à 110 ». C'est un
défaut du *dictionnaire*, pas du texte — le CRH peut être excellent.
→ *Destination : le réancrage (étage 2) trie et récupère. Lecture
utile : les fantômes de la section `informations` (Sexe, dates, noms —
le texte dit « Mme », le dictionnaire cite « Féminin ») sont un défaut de
contrat des métadonnées ; ceux de la section `diagnostics` se réancrent
presque tous.*

**`code absent du texte`** (`code_absent_texte`)
Un code du scénario n'a **aucune entrée** au dictionnaire : le modèle ne
l'a ni raconté ni annoté. L'échec le plus grave — le CRH n'exprime pas
son codage.
→ *Destination : RÉGÉNÉRATION (bloc correctif à formulation imposée,
tirée de la fiche du code).*

**`fidélité poids/taille`** (`fidelite_poids_taille`)
Le scénario fournit des valeurs exactes (52 kg, 155 cm) ; le texte doit
les restituer telles quelles. Échec = valeur absente ou modifiée.
→ *Destination : RÉGÉNÉRATION.*

**`service du scénario (X) absent du CR`** (`fidelite_service`)
Quand le scénario fournit une ligne « - Service : X », X doit apparaître
dans le CRH (casse, accents et espaces indifférents). Échec = le modèle
l'a remplacé ou développé (« CHIR.PLAST.RECONSTR. » → « Chirurgie
Plastique et Reconstructrice »).
→ *Destination : même famille que poids/taille — fidélité aux données
fournies.*

**`json invalide` / `fichier absent`**
Le retour du modèle n'était pas le JSON attendu, ou le fichier manque.
Quasi disparu depuis le prefill `{"CR": "` ; le chargeur répare les
défauts mineurs (et le signale en avertissement `json réparé`).

### Les AVERT (informations — jamais bloquants)

**`clés absentes du dictionnaire : X, Y`** — le gabarit attend certaines
clés d'informations (Dates, Service, Médecin, Traitements…) ; toutes ne
sont pas là. Indicatif : la complétude des *codes* a son propre contrôle.

**`clé orpheline`** — l'inverse : une entrée du dictionnaire porte un
code qui n'est **pas** au scénario (le modèle a codé en plus). Parfois
légitime — à l'œil du DIM.

**`mention IPP / Nom / Prénom / Date de naissance absente de
l'en-tête`** — mentions administratives attendues par le gabarit.
Avertissement (et non échec) tant que l'identité n'est pas fournie par le
scénario : on ne sanctionne pas le modèle pour une donnée qu'on ne lui
donne pas. Repassera en contrôle de fidélité le jour venu.

**`maladie chronique mentionnée : diabète (à confronter au scénario)`**
Le texte mentionne une maladie chronique surveillée (diabète, HTA,
fibrillation…) : vérifier qu'un code du scénario la justifie. Le garde
existe pour attraper la chronique *inventée* sans code ; quand le code
est là (I10 pour « hypertension »), c'est un faux positif bénin.

---

## Étage 2 — les sorties de `reancre_crh`

Le réancrage reprend chaque formulation fantôme et cherche dans le texte
un passage quasi identique (seuil 0,75) :

**`EXACTE`** — l'extrait cité est mot pour mot dans le texte.
L'annotation est parfaite. *(07 : 340/423, soit 80 %.)*

**`RÉANCRÉE (score)`** — un quasi-identique existe au-dessus du seuil :
l'extrait cité est **remplacé par l'extrait exact** du texte,
l'annotation est récupérée. *(07 : 53, soit 12 %.)*

**`ORPHELINE (meilleur score X)`** — rien d'assez proche : l'entrée sera
**supprimée à l'export** (tracée dans `supprimees`). Le score dit si
c'était loin (0,59 : vraiment rien) ou frôlé (0,73 : juste sous le
seuil — matière à relecture humaine). *(07 : 30, soit 7 %.)*

**La règle de lecture** : le chiffre brut des fantômes (83 sur le 07) ne
veut rien dire seul — après réancrage il devient 30 vraies orphelines,
dont la majorité en section `informations` (métadonnées), pas en
`diagnostics`.

---

## Étage 3 — les entrées juge (`entrees_juge.jsonl`)

Une ligne JSONL **par code de scénario** :
`{scenario, code, libelle, fiche, passages}` — les passages sont les
formulations *nettoyées* (exactes + réancrées) de ce code.

**La question posée au juge** (relecture manuelle aujourd'hui, encodeur
entraîné demain) : *au moins un de ces passages est-il un équivalent
direct du code ?* — réponse `preuve_directe: true/false`, avec le
`passage_retenu` qui devient l'ancre d'annotation du code.

Pourquoi cet étage existe : un extrait peut matcher **exactement** dans
le texte sans rien prouver. Cas d'école (07/0005) : le code Z99.1
(ventilation par masque nasal) a une entrée au dictionnaire dont
l'extrait est bien dans le texte — mais le texte ne parle **jamais** de
ventilation. Le contrôle mécanique passe ; seul un jugement d'équivalence
médicale voit que la preuve est fausse. Verdict `false` → régénération.

**Cas limites** : `passages: []` (toutes les preuves du code étaient
orphelines) → verdict `false` obligatoire ; `fiche: null` (pas de
référentiel pour juger) → relecture humaine.

---

## `verdict.txt` — l'ancien vérificateur LLM (optionnel, payant)

Un second appel au modèle qui relit le CRH selon **sa propre doctrine**
— il peut par exemple exiger qu'un « diagnostic principal soit clairement
identifié », ce qu'aucun CRH réel ne fait (c'est le travail du codeur).
Ses « NON CONFORME » se lisent avec distance. La chaîne mécanique +
juge le remplace ; il reste disponible comme contre-lecture ponctuelle.

---

## La grille de synthèse : qui va où

| Sortie | Gravité | Destination |
|---|---|---|
| gras | forme | nettoyage d'export |
| formulation fantôme | annotation | réancrage → export propre |
| orpheline (après réancrage) | annotation | supprimée à l'export, tracée |
| code absent du texte | contenu | **régénération** |
| fidélité poids/taille/service | contenu | **régénération** |
| verdict juge `preuve_directe: false` | contenu | **régénération** |
| clé orpheline, chroniques, mentions | information | œil du DIM |
| json invalide/réparé | forme | chargeur réparateur / re-run si besoin |

**En une phrase** : le gras et les fantômes se *nettoient*, les codes
absents et les infidélités se *régénèrent*, les avertissements se
*relisent* — et un bilan de 100 échecs peut être parfaitement sain une
fois chaque compteur rangé dans sa colonne.

## Documentation — le pipeline de données AP-HP

Les fichiers d'entrée et leurs producteurs, les transformations dans l'ordre
(substitution des DP, `agean`, racine, typologie, spécialité, enrichissement,
contrôle des fiches) et le contenu du scénario final sont décrits dans
[docs/pipeline_donnees_aphp.md](../docs/pipeline_donnees_aphp.md) — une seule
adresse fait foi, rien n'est recopié ici.
